# 实验室自动化中的优化问题、数学模型与求解器总结

# 0. 总览

| 序号 | 模型/问题类型 | 对应实验室问题 | 主要决策 | 典型目标 | 常见求解器 |
|---|---|---|---|---|---|
| 1 | 线性规划 LP | 试剂采购量、库存补货、资源用量分配、预算分配 | 连续采购量、资源分配量 | 成本最小、资源利用最大 | HiGHS、SciPy linprog、Gurobi、CPLEX |
| 2 | 整数规划 ILP / 0-1 规划 | 是否开启批次、是否启用设备、是否选择实验方案 | 0/1 决策、整数数量 | 固定成本最小、收益最大 | CBC、GLPK、SCIP、Gurobi、CPLEX |
| 3 | 混合整数线性规划 MILP | 样本分配到设备、设备启用、批次划分、采购固定成本 | 连续量 + 0/1 分配 | 总成本最小、产能满足 | CBC、SCIP、HiGHS、Gurobi、CPLEX |
| 4 | 指派问题 Assignment | 样本分配设备、孔位分配、任务分配仪器 | 一对一或一对多匹配 | 分配成本最小、质量最好 | OR-Tools、SciPy、Gurobi、CPLEX |
| 5 | 装箱问题 Bin Packing | 实验批次划分、96 孔板装载、PCR 批处理 | 样本放入哪个批次/板/转子 | 批次数最少、容量利用率最高 | OR-Tools CP-SAT、CBC、SCIP、Gurobi |
| 6 | 聚类 / 分组优化 | 相似样本分批、批次效应控制、分层随机化 | 样本分组 | 组内相似或组间均衡 | scikit-learn、OR-Tools、Pyomo、Gurobi |
| 7 | 约束满足问题 CSP | 孔板布局、QC 规则、对照布局、不兼容样本隔离 | 每个位置放什么 | 满足规则、冲突最少 | OR-Tools CP-SAT、python-constraint、MiniZinc |
| 8 | 调度 Scheduling | 今天哪些样本先做、哪些后做、日内任务排序 | 任务开始时间、顺序 | 延迟最小、完工时间最小 | OR-Tools CP-SAT、Timefold、Gurobi、CPLEX |
| 9 | 单机 / 并行机调度 | LC-MS 上机顺序、读板仪顺序、多台同类仪器处理 | 上机顺序、设备分配 | 总延迟最小、最大完工时间最小 | OR-Tools CP-SAT、Gurobi、CPLEX、启发式 |
| 10 | 作业车间调度 Job Shop | PCR / 孵育 / 离心 / 读板多步骤流程 | 每道工序在哪台设备何时做 | Makespan 最小、等待最少 | OR-Tools CP-SAT、IBM CP Optimizer、Gurobi |
| 11 | 资源受限项目调度 RCPSP | 多仪器共享、机械臂协同、人机协作 | 活动开始时间、资源占用 | 项目总时长最小、资源冲突最少 | OR-Tools CP-SAT、Timefold、CP Optimizer |
| 12 | TSP / Sequencing | 移液路径、枪头孔位访问顺序、机械臂站点访问 | 访问顺序 | 距离/时间最短 | OR-Tools Routing、LKH、NetworkX |
| 13 | VRP / Routing | 机械臂转运、样本设备间转运、冷链路径 | 多车辆/机械臂路径 | 总路程最短、超时最少 | OR-Tools Routing、LKH、Gurobi、CPLEX |
| 14 | 网络流 Network Flow | 自动化产线最大通量、多设备流量分配、瓶颈分析 | 样本流量 | 最大通量、最小费用 | NetworkX、OR-Tools、Gurobi、CPLEX |
| 15 | 库存优化 Inventory | 试剂采购、安全库存、补货批量、有效期 | 订货量、订货时点 | 持有成本 + 缺货成本最小 | Pyomo、SciPy、Gurobi、CPLEX |
| 16 | 随机优化 Stochastic | 需求不确定、设备故障、加工时间波动 | 不确定场景下的计划 | 期望成本最小、服务水平达标 | Pyomo、Gurobi、CPLEX、仿真工具 |
| 17 | 鲁棒优化 Robust | 故障、时间波动、消耗不确定下的稳健计划 | 保守决策、缓冲量 | 最坏情况损失最小 | Pyomo、CVXPY、Gurobi、MOSEK |
| 18 | 在线优化 / 滚动时域 | 故障重排程、插队样本、实时任务到达 | 当前窗口内的动态计划 | 实时可行、扰动最小 | OR-Tools CP-SAT、启发式、Timefold |
| 19 | 仿真优化 Simulation Optimization | 产线通量、瓶颈分析、设备配置 | 配置参数、规则参数 | 仿真指标最优 | SimPy + Optuna、Nevergrad、Ray Tune |
| 20 | 非线性规划 NLP | 试剂配方、温度/时间/浓度/pH 优化 | 连续实验参数 | 响应最大、偏差最小 | SciPy optimize、IPOPT、CasADi、MOSEK |
| 21 | 黑盒优化 | 无显式公式的实验效果优化、配方性能优化 | 实验候选点 | 实验效果最大 | Optuna、Nevergrad、DEAP、Ray Tune |
| 22 | 贝叶斯优化 | 少量实验预算下参数优化、主动学习实验设计 | 下一组实验条件 | 信息增益最大、响应最大 | BoTorch、Ax、scikit-optimize、Optuna |
| 23 | 多目标优化 | 时间、成本、质量、风险同时优化 | Pareto 解、权重方案 | Pareto 前沿最优 | pymoo、DEAP、BoTorch、Gurobi |
| 24 | DAG / Workflow Scheduling | 数据分析 pipeline、实验流程 DAG、CPU/GPU 调度 | DAG 任务分配与执行顺序 | 完成时间最小、资源利用率高 | Airflow、Prefect、Dagster、Nextflow |

---

# 1. 线性规划 LP

模型类型：
- LP，Linear Programming

特点：
- 决策变量通常是连续变量
- 目标函数是线性的
- 约束条件也是线性的
- 适合采购量、资源分配、预算分配等“数量连续可调”的问题
- 求解速度快，适合作为复杂模型的入门和子问题

现实例子：试剂采购量优化

实验室需要采购多种试剂，每种试剂有单价、最小需求量、冷库存储容量限制。要决定每种试剂采购多少，使总成本最小。

要决定：
- 每种试剂采购多少
- 每类冷库容量分配多少
- 不同实验项目之间如何分配预算

目标：
- 总采购成本最小
- 或在预算固定时满足尽可能多的需求

变量：
- \(x_i\)：试剂 \(i\) 的采购量
- \(c_i\)：试剂 \(i\) 的单位成本
- \(d_i\)：试剂 \(i\) 的最低需求量
- \(v_i\)：试剂 \(i\) 的单位冷库存储体积
- \(V\)：冷库总容量

数学模型：

$$
\min \sum_i c_i x_i
$$

约束：

$$
x_i \ge d_i,\quad \forall i
$$

$$
\sum_i v_i x_i \le V
$$

$$
x_i \ge 0,\quad \forall i
$$

约束解释：
- 每种试剂采购量必须满足最低需求
- 所有试剂占用的冷库空间不能超过容量
- 采购量不能为负

适用的实验室自动化场景：
- 试剂采购量优化
- 试剂库存补货
- 实验资源用量分配
- 项目预算分配
- 培养基、缓冲液、耗材的连续用量规划

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| SciPy linprog | 开源 | 小到中规模 LP，易用 |
| HiGHS | 开源 | 高性能 LP/MIP 求解器 |
| Gurobi | 非开源/商业 | 大规模工业级 LP |
| CPLEX | 非开源/商业 | 企业级 LP/MIP |

求解器选择建议：
- 小规模问题：SciPy linprog 或 HiGHS
- 大规模问题：HiGHS、Gurobi、CPLEX
- 开源优先：HiGHS
- 商业高性能：Gurobi、CPLEX、MOSEK

---

# 2. 整数规划 ILP / 0-1 规划

模型类型：
- ILP，Integer Linear Programming
- Binary Integer Programming，0-1 规划

特点：
- 变量必须取整数或 0/1
- 适合“是否选择”“是否启用”“是否安排”的决策
- 目标和约束仍是线性的
- 通常比 LP 难，但表达能力更强
- 可用于批次开启、设备启用、实验方案选择

现实例子：是否开启实验批次

实验室有若干候选实验批次，每个批次有固定启动成本和可处理样本集合。要决定今天开启哪些批次，使覆盖足够样本且成本最小。

要决定：
- 是否开启某个批次
- 是否启用某台设备
- 是否选择某种实验方案
- 是否安排某个复测任务

目标：
- 固定启动成本最小
- 覆盖样本数量最大
- 优先级收益最大

变量：
- \(y_b\)：是否开启批次 \(b\)，0 或 1
- \(a_{ib}\)：样本 \(i\) 是否可由批次 \(b\) 处理
- \(F_b\)：开启批次 \(b\) 的固定成本

数学模型：

$$
\min \sum_b F_b y_b
$$

约束：

$$
\sum_b a_{ib}y_b \ge 1,\quad \forall i
$$

$$
y_b \in \{0,1\},\quad \forall b
$$

约束解释：
- 每个样本至少被一个开启的批次覆盖
- 每个批次只能选择开启或不开启

适用的实验室自动化场景：
- 是否开启某个实验批次
- 是否启用某台设备
- 是否选择某个实验方案
- 是否安排某个复测任务
- 是否购买某个试剂包装规格

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| CBC | 开源 | 中小规模 ILP |
| GLPK | 开源 | 教学和小规模整数规划 |
| SCIP | 学术免费/部分商业限制 | 组合优化能力强 |
| Gurobi | 非开源/商业 | 大规模 ILP 性能强 |

求解器选择建议：
- 小规模：CBC、GLPK
- 中大规模：SCIP、Gurobi、CPLEX
- 若问题包含大量逻辑规则：也可考虑 CP-SAT

---

# 3. 混合整数线性规划 MILP

模型类型：
- MILP，Mixed Integer Linear Programming

特点：
- 一部分变量是连续变量
- 一部分变量是整数或 0/1 变量
- 目标和约束是线性的
- 适合同时包含“是否启用”和“分配多少”的问题
- 是实验室自动化中最常用的通用建模方式之一

现实例子：设备启用 + 样本分配

实验室有 3 台候选 PCR 仪，100 个样本。每台设备有固定启用成本、处理容量和样本处理成本。

要决定：
- 哪些 PCR 仪今天启用
- 每个样本由哪台 PCR 仪处理
- 每台 PCR 仪处理多少样本

目标：
- 设备启用固定成本 + 样本处理成本最小

变量：
- \(y_j\)：是否启用设备 \(j\)，0 或 1
- \(x_{ij}\)：样本 \(i\) 是否分配给设备 \(j\)，0 或 1
- \(F_j\)：设备 \(j\) 的启用固定成本
- \(c_{ij}\)：样本 \(i\) 在设备 \(j\) 上处理的成本
- \(K_j\)：设备 \(j\) 的处理容量

数学模型：

$$
\min \left(\sum_j F_j y_j + \sum_i \sum_j c_{ij}x_{ij}\right)
$$

约束：

$$
\sum_j x_{ij}=1,\quad \forall i
$$

$$
\sum_i x_{ij}\le K_j y_j,\quad \forall j
$$

$$
x_{ij}\in\{0,1\},\quad y_j\in\{0,1\}
$$

约束解释：
- 每个样本必须且只能分配给一台设备
- 每台设备处理量不能超过容量
- 只有启用了设备，才能给它分配样本

适用的实验室自动化场景：
- 样本分配到设备
- 设备启用与任务分配联合优化
- 批次划分与容量约束
- 试剂采购固定成本 + 变量成本
- 日内设备选择与任务分派

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| HiGHS | 开源 | 支持 LP/MIP，性能好 |
| CBC | 开源 | 常用开源 MILP |
| SCIP | 学术免费/部分商业限制 | 强组合优化能力 |
| Gurobi | 非开源/商业 | 大规模 MILP 首选之一 |
| CPLEX | 非开源/商业 | 企业级 MILP |

求解器选择建议：
- 小规模：PuLP + CBC、Pyomo + HiGHS
- 大规模：Gurobi、CPLEX、SCIP
- 实时要求强：可用启发式或 CP-SAT 替代精确 MILP
- 开源优先：HiGHS、CBC、SCIP

---

# 4. 指派问题 Assignment

模型类型：
- Assignment Problem
- 特殊的 0-1 线性规划
- 可看作二分图匹配问题

特点：
- 通常是一类对象分配给另一类对象
- 每个任务通常只能分配一次
- 每个资源有容量或最多接收一个任务
- 结构简单，求解很快
- 是样本-设备、样本-孔位、任务-仪器分配的基础模型

现实例子：样本分配到哪台设备

有 100 个样本和 5 台读板仪。不同样本在不同设备上的处理时间、等待时间或兼容性不同。

要决定：
- 每个样本分配到哪台设备
- 每个样本放到哪个孔位
- 每个实验任务由哪个仪器执行

目标：
- 总处理成本最小
- 总等待时间最小
- 设备负载尽量均衡

变量：
- \(x_{ij}\)：样本 \(i\) 是否分配给设备或孔位 \(j\)
- \(c_{ij}\)：分配成本
- \(K_j\)：设备或板位组 \(j\) 的容量

数学模型：

$$
\min \sum_i \sum_j c_{ij}x_{ij}
$$

约束：

$$
\sum_j x_{ij}=1,\quad \forall i
$$

$$
\sum_i x_{ij}\le K_j,\quad \forall j
$$

$$
x_{ij}\in\{0,1\}
$$

约束解释：
- 每个样本必须分配一次
- 每台设备或每类孔位不能超过容量
- 只能选择或不选择某个分配关系

适用的实验室自动化场景：
- 样本分配到哪台设备
- 样本分配到哪个孔位
- 实验任务分配给哪个仪器
- 技术员任务分派
- 数据分析任务分配给计算节点

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| SciPy linear_sum_assignment | 开源 | 标准匈牙利算法 |
| OR-Tools | 开源 | 可扩展到容量和复杂约束 |
| NetworkX | 开源 | 图匹配原型 |
| Gurobi | 非开源/商业 | 复杂指派 MILP |

求解器选择建议：
- 标准一对一指派：SciPy
- 带容量、兼容性、业务规则：OR-Tools CP-SAT 或 MILP
- 大规模工业场景：Gurobi、CPLEX

---

# 5. 装箱问题 Bin Packing

模型类型：
- Bin Packing
- Cutting / Packing
- 0-1 规划或 CP-SAT

特点：
- 将多个对象放入容量有限的容器
- 目标通常是使用容器数量最少
- 适合批次划分、孔板装载、PCR 批处理
- 是 NP-hard 问题，大规模时常用启发式
- 容量可以是孔数、体积、重量、时间窗口等

现实例子：96 孔板样本装载

实验室有 230 个样本，每块 96 孔板需要预留 8 个 QC 和对照孔，因此最多放 88 个样本。

要决定：
- 每个样本放入哪块板
- 需要开启多少块板
- 每块板是否满足 QC 和容量规则

目标：
- 使用孔板数量最少
- 每块板装载尽量均衡
- 批次间样本结构尽量一致

变量：
- \(x_{ib}\)：样本 \(i\) 是否放入批次/孔板 \(b\)
- \(y_b\)：是否启用批次/孔板 \(b\)
- \(w_i\)：样本 \(i\) 占用容量，通常为 1
- \(K_b\)：批次/孔板 \(b\) 的容量

数学模型：

$$
\min \sum_b y_b
$$

约束：

$$
\sum_b x_{ib}=1,\quad \forall i
$$

$$
\sum_i w_i x_{ib}\le K_b y_b,\quad \forall b
$$

$$
x_{ib},y_b\in\{0,1\}
$$

约束解释：
- 每个样本必须放入一个批次
- 每个批次容量不能超过限制
- 没启用的批次不能放样本

适用的实验室自动化场景：
- 实验批次划分
- 96 孔板样本装载
- PCR 仪批处理
- 离心机转子容量安排
- 冷冻盒样本存储安排

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| OR-Tools CP-SAT | 开源 | 适合装箱和逻辑约束 |
| CBC | 开源 | MILP 建模 |
| SCIP | 学术免费/部分商业限制 | 强组合优化 |
| Gurobi | 非开源/商业 | 大规模装箱 MILP |

求解器选择建议：
- 规则简单：First Fit Decreasing 启发式
- 带复杂业务规则：OR-Tools CP-SAT
- 需要证明最优：MILP + Gurobi/CPLEX/SCIP

---

# 6. 聚类 / 分组优化 Clustering

模型类型：
- Clustering
- Stratified Grouping
- Balanced Partitioning

特点：
- 根据样本特征进行分组
- 可以让相似样本同批，也可以让各批次分布均衡
- 常用于控制批次效应
- 既可用机器学习聚类，也可用整数规划做受约束分组
- 实验设计中常与随机化结合

现实例子：样本分层随机化

有 200 个临床样本，包含疾病状态、性别、年龄、采样中心等特征。希望分成 4 个实验批次，每批样本结构尽量相似。

要决定：
- 每个样本属于哪个批次
- 每个批次的病例/对照比例
- 每个批次的关键协变量分布

目标：
- 批次间特征分布差异最小
- 每个批次数量均衡
- 避免批次效应和混杂

变量：
- \(x_{ib}\)：样本 \(i\) 是否分配到批次 \(b\)
- \(z_b\)：批次 \(b\) 的特征均值
- \(f_i\)：样本 \(i\) 的某个数值特征
- \(K_b\)：批次容量

数学模型：

$$
\min \sum_b \left| z_b-\bar f \right|
$$

约束：

$$
z_b=\frac{1}{K_b}\sum_i f_i x_{ib},\quad \forall b
$$

$$
\sum_b x_{ib}=1,\quad \forall i
$$

$$
\sum_i x_{ib}=K_b,\quad \forall b
$$

$$
x_{ib}\in\{0,1\}
$$

约束解释：
- 每个样本只能属于一个批次
- 每个批次数量固定或接近固定
- 批次特征均值尽量接近总体均值

适用的实验室自动化场景：
- 相似样本分到同批或不同批
- 批次效应控制
- 实验条件分组
- 样本分层随机化
- 多板实验的样本均衡布局

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| scikit-learn | 开源 | K-means、层次聚类等快速原型 |
| OR-Tools CP-SAT | 开源 | 带容量和规则的分组 |
| Pyomo | 开源 | 自定义优化模型 |
| Gurobi | 非开源/商业 | 大规模受约束分组 |

求解器选择建议：
- 纯探索分析：scikit-learn
- 有硬约束：CP-SAT 或 MILP
- 需要分层随机化和均衡：MILP/CP-SAT 更合适

---

# 7. 约束满足问题 CSP

模型类型：
- CSP，Constraint Satisfaction Problem
- CP，Constraint Programming
- CP-SAT

特点：
- 重点不是优化目标，而是找到满足所有规则的解
- 适合复杂逻辑规则、兼容性、排斥关系
- 可加入软约束并最小化违规惩罚
- 非常适合孔板布局和合规质控规则检查
- 比 MILP 更自然地表达“相邻不能放”“固定间隔”等规则

现实例子：孔板布局设计

96 孔板中需要安排样本、阳性对照、阴性对照、空白孔和 QC。要求对照分布均匀，不兼容样本不能相邻，边缘孔尽量避免关键样本。

要决定：
- 每个孔位放哪个样本或对照
- QC 样本出现在哪些孔位
- 阳性/阴性对照如何分布

目标：
- 满足所有硬性合规规则
- 最小化软规则违规
- 降低边缘效应和交叉污染风险

变量：
- \(x_{ip}\)：样本或对照 \(i\) 是否放在孔位 \(p\)
- \(v_r\)：规则 \(r\) 是否违规
- \(N(p)\)：孔位 \(p\) 的相邻孔集合

数学模型：

$$
\min \sum_r \lambda_r v_r
$$

约束：

$$
\sum_p x_{ip}=1,\quad \forall i
$$

$$
\sum_i x_{ip}\le 1,\quad \forall p
$$

不兼容样本 \(i,k\) 不能相邻：

$$
x_{ip}+x_{kq}\le 1,\quad \forall q\in N(p)
$$

QC 固定数量：

$$
\sum_{i\in QC}\sum_p x_{ip}=Q
$$

约束解释：
- 每个样本或对照必须放置一次
- 每个孔位最多放一个对象
- 不兼容样本不能相邻
- QC、阳性对照、阴性对照数量和位置需满足规则

适用的实验室自动化场景：
- 孔板布局设计
- 合规质控规则检查
- 阳性/阴性对照布局
- 不兼容样本隔离
- 交叉污染风险控制

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| OR-Tools CP-SAT | 开源 | 复杂逻辑和优化都支持 |
| python-constraint | 开源 | 小规模 CSP 原型 |
| MiniZinc | 开源 | 约束建模语言 |
| IBM CP Optimizer | 非开源/商业 | 工业级 CP/调度 |

求解器选择建议：
- 小规模规则验证：python-constraint
- 孔板布局生产优化：OR-Tools CP-SAT
- 企业级复杂排程和约束：IBM CP Optimizer

---

# 8. 调度问题 Scheduling

模型类型：
- Scheduling
- MILP / CP-SAT / 启发式算法

特点：
- 决定任务先后顺序和开始时间
- 常带有截止时间、优先级、设备容量
- 是“今天哪些样本先做、哪些后做”的核心模型
- 可建成 MILP，也可用 CP-SAT 表达
- 实际系统中常配合滚动时域优化

现实例子：日内样本处理排序

今天有 80 个样本，每个样本有处理时长、优先级和截止时间。实验室需要安排处理顺序，减少延迟。

要决定：
- 哪些样本先做
- 每个任务何时开始
- 是否允许某些低优先级样本延期

目标：
- 总延迟最小
- 高优先级样本按时完成
- 最大完工时间最小

变量：
- \(s_i\)：任务 \(i\) 的开始时间
- \(C_i\)：任务 \(i\) 的完成时间
- \(T_i\)：任务 \(i\) 的延迟
- \(p_i\)：任务 \(i\) 的处理时间
- \(d_i\)：任务 \(i\) 的截止时间

数学模型：

$$
\min \sum_i w_i T_i
$$

约束：

$$
C_i=s_i+p_i,\quad \forall i
$$

$$
T_i\ge C_i-d_i,\quad T_i\ge 0,\quad \forall i
$$

单设备不重叠可用顺序变量 \(z_{ij}\) 表示：

$$
s_i+p_i\le s_j+M(1-z_{ij}),\quad \forall i<j
$$

$$
s_j+p_j\le s_i+Mz_{ij},\quad \forall i<j
$$

约束解释：
- 完成时间等于开始时间加处理时长
- 延迟是超过截止时间的部分
- 同一设备上任意两个任务不能重叠

适用的实验室自动化场景：
- 今天哪些样本先做、哪些后做
- 日内任务排序
- 样本优先级和截止时间管理
- 急诊样本插队
- 复测任务安排

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| OR-Tools CP-SAT | 开源 | 调度建模能力强 |
| Timefold | 开源/商业版本并存 | 适合业务排程 |
| Gurobi | 非开源/商业 | MILP 调度 |
| IBM CP Optimizer | 非开源/商业 | 高级调度约束 |

求解器选择建议：
- 标准调度：OR-Tools CP-SAT
- 复杂业务排程：Timefold、CP Optimizer
- 需要和成本、分配联合优化：MILP
- 实时场景：启发式 + 滚动重排程

---

# 9. 单机 / 并行机调度

模型类型：
- Single Machine Scheduling
- Parallel Machine Scheduling

特点：
- 单机调度：所有任务排在一台设备上
- 并行机调度：多台相同或相似设备并行处理
- 适合 LC-MS、读板仪、测序仪等上机顺序问题
- 目标常为最小化最大完工时间、总延迟或加权完工时间
- 可扩展为设备速度不同的 unrelated parallel machine scheduling

现实例子：多台读板仪并行处理

实验室有 4 台读板仪，120 块板需要读数。不同板有不同截止时间，某些板只能使用特定读板仪。

要决定：
- 每块板分配到哪台读板仪
- 每台读板仪上的上机顺序
- 每块板的开始和完成时间

目标：
- 最大完工时间最小
- 总延迟最小
- 各设备负载均衡

变量：
- \(x_{ij}\)：任务 \(i\) 是否分配给设备 \(j\)
- \(s_i\)：任务 \(i\) 的开始时间
- \(C_i\)：任务 \(i\) 的完成时间
- \(C_{\max}\)：最大完工时间

数学模型：

$$
\min C_{\max}
$$

约束：

$$
\sum_j x_{ij}=1,\quad \forall i
$$

$$
C_i=s_i+p_i,\quad \forall i
$$

$$
C_i\le C_{\max},\quad \forall i
$$

同一设备上任务不重叠：

$$
s_i+p_i\le s_k+M(1-z_{ikj}),\quad \forall i,k,j
$$

约束解释：
- 每个任务分配到一台设备
- 完工时间不能超过最大完工时间
- 同一设备上的任务不能重叠
- 设备兼容性可通过 \(x_{ij}\le a_{ij}\) 表达

适用的实验室自动化场景：
- 读板仪上机顺序
- LC-MS 上机顺序
- 多台同类仪器并行处理
- 测序仪排队
- 显微镜扫描任务排程

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| OR-Tools CP-SAT | 开源 | Interval 变量适合调度 |
| Timefold | 开源/商业版本并存 | 业务排程 |
| Gurobi | 非开源/商业 | 精确 MILP |
| CPLEX | 非开源/商业 | 企业级调度 MILP |

求解器选择建议：
- 几十到几百任务：CP-SAT
- 大规模但允许近似：启发式、局部搜索
- 商业高性能：Gurobi、CPLEX、CP Optimizer

---

# 10. 作业车间调度 Job Shop Scheduling

模型类型：
- Job Shop Scheduling
- Flow Shop Scheduling
- CP-SAT / MILP

特点：
- 每个样本或作业包含多个工序
- 工序必须按固定顺序执行
- 不同工序需要不同设备
- 适合 PCR / 孵育 / 离心 / 读板等多步骤流程
- 比单机调度复杂，常用 CP-SAT 或 CP Optimizer

现实例子：PCR / 孵育 / 离心 / 读板流程排程

每个样本需要依次经过提取、PCR、孵育、离心、读板。每类设备数量有限，且工序之间存在先后依赖。

要决定：
- 每个样本每道工序的开始时间
- 每道工序使用哪台设备
- 不同样本在设备上的顺序

目标：
- 最大完工时间最小
- 等待时间最小
- 高优先级样本延迟最小

变量：
- \(s_{io}\)：样本 \(i\) 的工序 \(o\) 开始时间
- \(C_{io}\)：样本 \(i\) 的工序 \(o\) 完成时间
- \(p_{io}\)：工序处理时间
- \(x_{iom}\)：工序 \(o\) 是否分配给设备 \(m\)

数学模型：

$$
\min C_{\max}
$$

约束：

$$
C_{io}=s_{io}+p_{io},\quad \forall i,o
$$

工序先后关系：

$$
s_{i,o+1}\ge C_{io},\quad \forall i,o
$$

设备分配：

$$
\sum_{m\in M_o}x_{iom}=1,\quad \forall i,o
$$

最大完工时间：

$$
C_{i,O_i}\le C_{\max},\quad \forall i
$$

约束解释：
- 每个工序完成后才能进入下一工序
- 每道工序必须分配到兼容设备
- 同一设备同一时间不能处理多个工序
- 最后一道工序的完成时间决定样本完工时间

适用的实验室自动化场景：
- PCR / 孵育 / 离心 / 读板流程排程
- 多步骤实验流程
- 每个样本按固定工艺路线经过多个设备
- 自动化工作站与外围设备协同
- 复杂检测流程排程

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| OR-Tools CP-SAT | 开源 | Job Shop 示例成熟 |
| IBM CP Optimizer | 非开源/商业 | 工业级调度强 |
| Gurobi | 非开源/商业 | MILP 精确建模 |
| Timefold | 开源/商业版本并存 | 业务规则排程 |

求解器选择建议：
- 优先选择 CP-SAT 表达工序和设备不重叠
- 复杂日历、维护窗口、批处理设备可用 CP Optimizer
- 与成本、采购、库存联动时可用 MILP

---

# 11. 资源受限项目调度 RCPSP

模型类型：
- RCPSP，Resource-Constrained Project Scheduling Problem

特点：
- 任务之间有前后依赖
- 多种共享资源容量有限
- 资源可包括机械臂、离心机、孵育器、人工、试剂
- 适合自动化产线中多设备协同
- 比 Job Shop 更强调多资源同时占用

现实例子：机械臂、孵育器、离心机协同

一个实验流程包含加样、转运、孵育、离心、读板等活动。某些活动需要同时占用机械臂和设备。

要决定：
- 每个活动何时开始
- 每个活动占用哪些资源
- 多个样本流程如何交错执行

目标：
- 整体流程完成时间最小
- 资源冲突最少
- 机械臂空闲和等待时间最小

变量：
- \(s_i\)：活动 \(i\) 的开始时间
- \(C_i\)：活动 \(i\) 的完成时间
- \(r_{ik}\)：活动 \(i\) 对资源 \(k\) 的需求量
- \(R_k\)：资源 \(k\) 的容量

数学模型：

$$
\min C_{\max}
$$

约束：

$$
C_i=s_i+p_i,\quad \forall i
$$

前后依赖：

$$
s_j\ge C_i,\quad \forall (i,j)\in E
$$

资源容量：

$$
\sum_{i: t\in[s_i,C_i)} r_{ik}\le R_k,\quad \forall k,t
$$

$$
C_i\le C_{\max},\quad \forall i
$$

约束解释：
- 有依赖关系的任务必须按顺序执行
- 任一时间点资源使用量不能超过容量
- 项目完成时间由最后完成的任务决定

适用的实验室自动化场景：
- 多台仪器排队和共享
- 机械臂、孵育器、离心机等共享资源协同
- 人机协作任务调度
- 自动化工作站多模块协同
- 设备维护窗口下的计划

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| OR-Tools CP-SAT | 开源 | 支持累计资源约束 |
| Timefold | 开源/商业版本并存 | 复杂业务排程 |
| IBM CP Optimizer | 非开源/商业 | RCPSP 能力强 |
| Gurobi | 非开源/商业 | MILP 资源调度 |

求解器选择建议：
- 多资源调度优先 CP-SAT 或 CP Optimizer
- 有复杂软约束和业务偏好：Timefold
- 需要全局最优证明：MILP/CP 精确求解

---

# 12. 路径优化 TSP / Sequencing

模型类型：
- TSP，Traveling Salesman Problem
- Sequencing
- Shortest Hamiltonian Path/Cycle

特点：
- 决定访问多个点的顺序
- 每个点访问一次
- 目标是总距离或总时间最短
- 适合移液枪头访问孔位顺序、机械臂站点访问顺序
- 若有多个载具、容量、时间窗，则扩展为 VRP

现实例子：移液路径优化

自动移液工作站需要访问 96 孔板中的多个孔位。枪头移动时间与孔位距离相关，需要找到访问顺序，使总移动距离最短。

要决定：
- 枪头先访问哪个孔
- 后访问哪个孔
- 是否按行、列或优化路径访问

目标：
- 移动距离最短
- 移液总时间最短
- 交叉污染风险最低

变量：
- \(x_{ij}\)：是否从孔位 \(i\) 直接移动到孔位 \(j\)
- \(d_{ij}\)：孔位 \(i\) 到孔位 \(j\) 的距离
- \(u_i\)：消除子回路的辅助变量

数学模型：

$$
\min \sum_i\sum_j d_{ij}x_{ij}
$$

约束：

$$
\sum_j x_{ij}=1,\quad \forall i
$$

$$
\sum_i x_{ij}=1,\quad \forall j
$$

MTZ 子回路消除：

$$
u_i-u_j+n x_{ij}\le n-1,\quad \forall i\ne j,\ i,j\ne 0
$$

$$
x_{ij}\in\{0,1\}
$$

约束解释：
- 每个孔位必须离开一次
- 每个孔位必须进入一次
- 子回路消除约束保证形成一条完整路径
- 若考虑污染，可禁止某些访问顺序

适用的实验室自动化场景：
- 移液路径优化
- 枪头访问孔位顺序
- 机械臂访问多个站点的顺序
- 显微镜扫描视野顺序
- 自动取样器进样顺序

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| OR-Tools Routing | 开源 | TSP/VRP 建模方便 |
| NetworkX | 开源 | 图算法和原型验证 |
| LKH | 通常可免费用于研究，许可需核查 | 高质量 TSP 启发式 |
| Gurobi | 非开源/商业 | TSP MILP 精确求解 |

求解器选择建议：
- 小规模精确求解：MILP
- 中大规模路径：OR-Tools Routing、LKH
- 快速原型：NetworkX
- 实时移液路径：启发式或预计算路径表

---

# 13. 车辆路径问题 VRP / Routing

模型类型：
- VRP，Vehicle Routing Problem
- Routing with Capacity / Time Windows

特点：
- 多个载具或机械臂执行转运任务
- 可包含容量、时间窗、取送货、冷链限制
- 是 TSP 的扩展
- 适合机械臂转运顺序和样本在设备间移动
- 工业系统中常与调度问题耦合

现实例子：机械臂转运顺序

自动化实验室有 2 个机械臂，需要在样本架、离心机、PCR 仪、读板仪之间搬运样本。不同样本有最晚转运时间。

要决定：
- 每个机械臂负责哪些转运任务
- 每个机械臂访问站点的顺序
- 每个样本何时被转运

目标：
- 总转运距离最短
- 超过时间窗的样本最少
- 机械臂利用率均衡

变量：
- \(x_{ijk}\)：机械臂 \(k\) 是否从节点 \(i\) 到节点 \(j\)
- \(t_{ik}\)：机械臂 \(k\) 到达节点 \(i\) 的时间
- \(q_i\)：节点 \(i\) 的样本数量或载荷
- \(Q_k\)：机械臂 \(k\) 的容量

数学模型：

$$
\min \sum_k\sum_i\sum_j d_{ij}x_{ijk}
$$

约束：

$$
\sum_k\sum_j x_{ijk}=1,\quad \forall i
$$

$$
\sum_i q_i x_{ijk}\le Q_k,\quad \forall k
$$

时间窗：

$$
a_i\le t_{ik}\le b_i,\quad \forall i,k
$$

路径连续性：

$$
\sum_i x_{ihk}=\sum_j x_{hjk},\quad \forall h,k
$$

约束解释：
- 每个任务必须由一个机械臂服务一次
- 机械臂载荷不能超过容量
- 访问时间必须满足时间窗
- 每个机械臂的路径必须连续

适用的实验室自动化场景：
- 机械臂转运顺序
- 样本在多个设备间转运
- 冷链样本路径规划
- 移动机器人配送耗材
- 自动仓储取样路径

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| OR-Tools Routing | 开源 | VRP、时间窗、容量支持好 |
| LKH | 通常可免费用于研究，许可需核查 | 大规模路径启发式 |
| Gurobi | 非开源/商业 | 精确 VRP MILP |
| CPLEX | 非开源/商业 | 商业级路径优化 |

求解器选择建议：
- TSP：单载具访问顺序
- VRP：多载具、容量、时间窗
- Sequencing：只关注顺序，不一定有空间路径
- Routing：更广义的路径和网络移动问题

---

# 14. 网络流 Network Flow

模型类型：
- Network Flow
- Max Flow / Min Cost Flow

特点：
- 将实验系统抽象为节点和边
- 节点表示设备、缓冲区、工位
- 边表示样本流转通道
- 适合最大通量、流量分配和瓶颈分析
- 线性结构强，求解速度快

现实例子：自动化产线最大通量

一条自动化产线包括样本输入、提取、PCR、读板和输出。每个设备有处理能力，设备之间有传输能力限制。

要决定：
- 每条路径上分配多少样本流
- 哪些设备成为瓶颈
- 是否需要增加设备容量

目标：
- 单位时间样本通量最大
- 或在满足需求下总流转成本最小

变量：
- \(f_{ij}\)：从节点 \(i\) 到节点 \(j\) 的样本流量
- \(u_{ij}\)：边容量
- \(c_{ij}\)：单位流量成本

数学模型：

最大流：

$$
\max \sum_j f_{sj}
$$

约束：

$$
0\le f_{ij}\le u_{ij},\quad \forall (i,j)
$$

流量守恒：

$$
\sum_i f_{iv}=\sum_j f_{vj},\quad \forall v\ne s,t
$$

最小费用流：

$$
\min \sum_{(i,j)} c_{ij}f_{ij}
$$

约束解释：
- 每条通道流量不能超过容量
- 中间节点流入量等于流出量
- 源点输出代表进入系统的样本量
- 汇点输入代表完成处理的样本量

适用的实验室自动化场景：
- 自动化产线最大通量
- 多设备串并联样本流分配
- 瓶颈分析
- 样本缓冲区容量设计
- 设备扩容评估

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| NetworkX | 开源 | 图算法原型方便 |
| OR-Tools | 开源 | 最小费用流支持 |
| Gurobi | 非开源/商业 | 大规模线性网络优化 |
| CPLEX | 非开源/商业 | 企业级网络流模型 |

求解器选择建议：
- 原型分析：NetworkX
- 工程应用：OR-Tools
- 与库存、调度、投资决策耦合：MILP + Gurobi/CPLEX

---

# 15. 库存优化 Inventory Optimization

模型类型：
- Inventory Optimization
- EOQ / Base-stock / Lot-sizing
- MILP / Stochastic Inventory

特点：
- 决定何时订货、订多少
- 需要权衡采购成本、持有成本、缺货风险
- 实验室还需考虑有效期、冷链容量、批号追踪
- 可确定性建模，也可随机需求建模
- 与 LIMS、ERP、仓储系统强相关

现实例子：试剂采购和安全库存

实验室每周消耗 qPCR 试剂，需求有波动。试剂有固定订货成本、单位成本、冷库存储成本和有效期。

要决定：
- 每个周期采购多少试剂
- 安全库存设多少
- 何时触发补货
- 是否拆分采购批次以避免过期

目标：
- 采购成本 + 持有成本 + 缺货惩罚 + 过期损失最小

变量：
- \(q_t\)：周期 \(t\) 的订货量
- \(I_t\)：周期 \(t\) 末库存
- \(D_t\)：周期 \(t\) 的需求
- \(B_t\)：周期 \(t\) 的缺货量
- \(y_t\)：周期 \(t\) 是否下单

数学模型：

$$
\min \sum_t \left(Fy_t+cq_t+hI_t+pB_t\right)
$$

约束：

$$
I_t=I_{t-1}+q_t-D_t+B_t,\quad \forall t
$$

$$
q_t\le My_t,\quad \forall t
$$

$$
I_t\le V,\quad \forall t
$$

$$
q_t,I_t,B_t\ge 0,\quad y_t\in\{0,1\}
$$

约束解释：
- 库存由上期库存、采购量和需求共同决定
- 有采购才产生订货量
- 库存不能超过冷库容量
- 缺货量用于表示服务水平不足

适用的实验室自动化场景：
- 试剂采购和库存
- 安全库存
- 补货批量
- 有效期和冷链容量
- 高价值抗体、酶、试剂盒管理

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| Pyomo | 开源 | 库存 MILP 建模灵活 |
| SciPy | 开源 | 简单连续库存优化 |
| HiGHS | 开源 | LP/MIP 求解 |
| Gurobi | 非开源/商业 | 大规模多周期库存 |

求解器选择建议：
- 简单 EOQ：公式法或 SciPy
- 多周期、多容量、多有效期：MILP
- 随机需求：随机优化或仿真优化

---

# 16. 随机优化 Stochastic Optimization

模型类型：
- Stochastic Optimization
- Scenario-based Optimization
- Two-stage Stochastic Programming

特点：
- 显式考虑不确定性
- 不确定因素包括需求、样本到达、设备故障、加工时间
- 常用场景集表示未来可能情况
- 目标通常是期望成本最小或服务水平达标
- 适合计划层决策，不一定适合毫秒级实时控制

现实例子：随机需求下试剂采购

未来一周样本量不确定，有低、中、高三种需求场景。实验室希望先采购一部分试剂，未来根据实际需求补货或承担缺货惩罚。

要决定：
- 第一阶段先采购多少
- 不同需求场景下如何补货
- 安全库存水平

目标：
- 期望总成本最小

变量：
- \(q\)：第一阶段采购量
- \(r_\omega\)：场景 \(\omega\) 下的追加采购量
- \(B_\omega\)：场景 \(\omega\) 下的缺货量
- \(P_\omega\)：场景概率
- \(D_\omega\)：场景需求

数学模型：

$$
\min cq+\sum_{\omega}P_\omega\left(c_r r_\omega+pB_\omega\right)
$$

约束：

$$
q+r_\omega+B_\omega\ge D_\omega,\quad \forall \omega
$$

$$
q,r_\omega,B_\omega\ge 0
$$

约束解释：
- 每个场景下，初始采购、追加采购和缺货共同覆盖需求
- 目标按场景概率计算期望成本
- 可加入服务水平约束，例如缺货概率不超过阈值

适用的实验室自动化场景：
- 试剂需求不确定
- 样本到达不确定
- 设备故障概率
- 加工时间波动
- 实验失败率不确定

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| Pyomo | 开源 | 场景随机规划建模 |
| HiGHS | 开源 | 求解线性场景模型 |
| Gurobi | 非开源/商业 | 大规模随机 MILP |
| CPLEX | 非开源/商业 | 企业级随机优化模型 |

求解器选择建议：
- 场景数量少：Pyomo + HiGHS/CBC
- 场景数量多：分解算法或商业求解器
- 若系统动态强：结合在线优化和仿真优化

---

# 17. 鲁棒优化 Robust Optimization

模型类型：
- Robust Optimization

特点：
- 不依赖精确概率分布
- 假设不确定参数落在某个集合内
- 优化最坏情况或保守可行性
- 适合设备故障、时间波动、试剂消耗不确定
- 通常比随机优化更保守

现实例子：处理时间波动下的稳健排程

PCR 处理时间可能在标称值上下浮动。实验室希望排程即使在处理时间变长时仍不大面积延误。

要决定：
- 每个任务的开始时间
- 是否预留缓冲时间
- 哪些关键任务优先安排

目标：
- 最坏情况下总延迟最小
- 或计划在所有不确定场景下都可行

变量：
- \(s_i\)：任务 \(i\) 的开始时间
- \(p_i\)：标称处理时间
- \(\Delta_i\)：最大处理时间扰动
- \(T_i\)：延迟

数学模型：

$$
\min \sum_i w_i T_i
$$

鲁棒完成时间：

$$
C_i=s_i+p_i+\Delta_i
$$

截止约束：

$$
T_i\ge C_i-d_i,\quad T_i\ge 0
$$

不重叠约束：

$$
s_i+p_i+\Delta_i\le s_j+M(1-z_{ij})
$$

约束解释：
- 使用最坏处理时间进行排程
- 任务之间即使发生时间波动也不重叠
- 延迟按最坏情况计算

适用的实验室自动化场景：
- 设备故障情况下仍能执行的计划
- 处理时间波动下的稳健排程
- 试剂消耗不确定下的稳健采购
- 样本延迟到达下的缓冲计划
- 冷链窗口不确定时的稳健路径

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| Pyomo | 开源 | 鲁棒 MILP 建模 |
| CVXPY | 开源 | 凸鲁棒优化方便 |
| Gurobi | 非开源/商业 | 大规模鲁棒 MILP |
| MOSEK | 非开源/商业 | 凸优化和锥优化强 |

求解器选择建议：
- 不确定性有概率：优先随机优化
- 不确定性无可靠概率：优先鲁棒优化
- 实时异常响应：鲁棒计划 + 在线重排程

---

# 18. 在线优化 / 滚动时域优化

模型类型：
- Online Optimization
- Rolling Horizon Optimization
- Model Predictive Scheduling

特点：
- 决策不是一次性完成，而是随时间不断更新
- 适合故障、插队样本、实时任务到达
- 每次只优化未来一个时间窗口
- 可减少大规模全局调度的计算压力
- 生产系统中非常常见

现实例子：设备故障后的重新排程

上午计划已生成，但一台离心机突然故障，并新增 20 个急诊样本。系统需要在不完全打乱已有计划的前提下重新排程。

要决定：
- 当前正在执行的任务是否保持不变
- 未开始任务如何重新排序
- 急诊样本插入到哪些设备和时间段
- 是否调用备用设备

目标：
- 新计划可行
- 高优先级样本延迟最小
- 对原计划扰动最小

变量：
- \(s_i^{new}\)：任务 \(i\) 的新开始时间
- \(s_i^{old}\)：任务 \(i\) 的原计划开始时间
- \(x_{ij}\)：任务 \(i\) 是否分配给设备 \(j\)
- \(T_i\)：任务延迟

数学模型：

$$
\min \sum_i w_iT_i+\lambda\sum_i |s_i^{new}-s_i^{old}|
$$

约束：

$$
s_i^{new}\ge t_{now},\quad \forall i\in U
$$

$$
x_{ij}=0,\quad \forall j\in \text{FailedDevices}
$$

$$
T_i\ge s_i^{new}+p_i-d_i,\quad T_i\ge 0
$$

约束解释：
- 已经开始或完成的任务通常冻结
- 未开始任务可重新安排
- 故障设备不能再分配任务
- 新计划尽量减少对原计划的扰动

适用的实验室自动化场景：
- 故障情况下重新排程
- 动态插队样本
- 实时任务到达
- 战略调度 + 战术调度二层架构
- 试剂缺货后的计划修复

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| OR-Tools CP-SAT | 开源 | 快速重排程 |
| Timefold | 开源/商业版本并存 | 业务排程与增量求解 |
| OptaPlanner | 开源，生态已向 Timefold 等方向变化 | 规划调度历史成熟 |
| Gurobi | 非开源/商业 | 滚动 MILP 优化 |

求解器选择建议：
- 实时性强：启发式、局部搜索、CP-SAT 限时求解
- 计划层优化：MILP/CP 完整模型
- 生产系统：冻结已执行任务，滚动优化未来窗口

---

# 19. 仿真优化 Simulation Optimization

模型类型：
- Simulation Optimization
- Discrete Event Simulation + Optimization

特点：
- 当系统太复杂难以写出显式公式时，用仿真评估方案
- 优化算法负责提出方案，仿真模型返回指标
- 适合自动化产线通量、瓶颈分析、设备配置评估
- 可以模拟排队、故障、维护、随机到达
- 常与 SimPy、AnyLogic、FlexSim、优化算法结合

现实例子：自动化产线最大通量

实验室计划增加一台机械臂或一台读板仪。通过离散事件仿真评估不同设备配置下的日通量、等待时间和瓶颈。

要决定：
- 每类设备配置数量
- 缓冲区容量
- 调度规则参数
- 是否增加备用设备

目标：
- 日通量最大
- 平均等待时间最小
- 设备投资成本可控

变量：
- \(x_k\)：设备类型 \(k\) 的数量
- \(b_l\)：缓冲区 \(l\) 的容量
- \(\theta\)：调度规则参数
- \(Y(x,b,\theta)\)：仿真输出通量

数学模型：

$$
\max \mathbb{E}[Y(x,b,\theta)]-\sum_k c_kx_k
$$

约束：

$$
\sum_k c_kx_k\le B
$$

$$
x_k\in\mathbb{Z}_+,\quad b_l\in\mathbb{Z}_+
$$

约束解释：
- 决策变量是设备配置和控制规则
- 目标由仿真模型估计，通常有随机噪声
- 总投资不能超过预算

适用的实验室自动化场景：
- 自动化产线最大通量
- 瓶颈分析
- 设备配置优化
- SimPy 离散事件仿真 + 优化算法
- 排队、故障、维护策略评估

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| SimPy | 开源 | Python 离散事件仿真 |
| Optuna | 开源 | 参数搜索方便 |
| Nevergrad | 开源 | 黑盒优化算法丰富 |
| Ray Tune | 开源 | 分布式实验搜索 |

求解器选择建议：
- 小规模参数：网格搜索或随机搜索
- 中等复杂黑盒：Optuna、Nevergrad
- 大量仿真并行：Ray Tune
- 高成本仿真：贝叶斯优化

---

# 20. 非线性规划 NLP

模型类型：
- NLP，Nonlinear Programming

特点：
- 目标函数或约束包含非线性关系
- 变量通常是连续变量
- 适合温度、时间、浓度、pH 等实验参数优化
- 如果函数可微，可用梯度法
- 若非凸，可能只能保证局部最优

现实例子：试剂配方优化

实验室需要优化缓冲液配方，变量包括盐浓度、pH、酶浓度和反应时间，目标是最大化信号强度并控制背景噪声。

要决定：
- 各组分浓度
- 反应温度
- 反应时间
- pH 值

目标：
- 实验响应最大
- 背景噪声最小
- 配方偏离标准范围最小

变量：
- \(x_1\)：盐浓度
- \(x_2\)：pH
- \(x_3\)：酶浓度
- \(x_4\)：反应时间
- \(f(x)\)：实验响应模型

数学模型：

$$
\max f(x_1,x_2,x_3,x_4)
$$

约束：

$$
l_i\le x_i\le u_i,\quad \forall i
$$

$$
g_m(x)\le 0,\quad \forall m
$$

若最小化偏差：

$$
\min \left(y^*-f(x)\right)^2+\lambda\sum_i (x_i-x_i^0)^2
$$

约束解释：
- 每个实验参数有上下限
- 某些组合可能因安全性或稳定性不可行
- 目标可以是响应最大，也可以是接近目标值

适用的实验室自动化场景：
- 试剂配方优化
- 实验参数连续优化
- 温度、时间、浓度、pH 优化
- 反应动力学参数拟合
- 光学读数校准

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| SciPy optimize | 开源 | 常用 NLP 原型 |
| IPOPT | 开源 | 大规模连续非线性优化 |
| CasADi | 开源 | 自动微分和最优控制 |
| MOSEK | 非开源/商业 | 凸优化和锥优化强 |

求解器选择建议：
- 函数可微且成本低：SciPy、IPOPT
- 非凸且多峰：多起点优化、全局优化或黑盒优化
- 实验成本高：贝叶斯优化更合适

---

# 21. 黑盒优化 Black-box Optimization

模型类型：
- Black-box Optimization
- Derivative-free Optimization

特点：
- 不知道显式数学公式
- 只能通过实验或仿真获得输出
- 可能有噪声、成本高、失败实验
- 可用随机搜索、进化算法、贝叶斯优化、CMA-ES
- 适合复杂生物实验响应优化

现实例子：无显式公式的配方性能优化

某细胞培养基配方的性能只能通过实际培养实验测得。变量包括多种添加剂浓度，目标是提高细胞活率和产量。

要决定：
- 下一批实验测试哪些配方
- 每个配方的组分浓度
- 是否探索新区域或利用已知好区域

目标：
- 实验性能最大
- 失败率最低
- 实验次数尽量少

变量：
- \(x\)：实验条件或配方向量
- \(y=f(x)+\epsilon\)：实验观测响应
- \(\mathcal{X}\)：可行实验空间

数学模型：

$$
x^*=\arg\max_{x\in\mathcal{X}} f(x)
$$

但 \(f(x)\) 不显式可得，只能通过实验获得：

$$
y_t=f(x_t)+\epsilon_t
$$

约束：

$$
l_i\le x_i\le u_i,\quad \forall i
$$

$$
x\in \mathcal{X}_{safe}
$$

约束解释：
- 每个配方参数有实验安全范围
- 黑盒函数只能通过实验评估
- 观测结果可能有噪声

适用的实验室自动化场景：
- 无显式公式的实验效果优化
- 配方性能优化
- 复杂生物实验响应优化
- 自动化实验平台闭环优化
- 高通量筛选策略优化

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| Optuna | 开源 | 易用，适合超参数和实验搜索 |
| Nevergrad | 开源 | 无导数优化算法丰富 |
| DEAP | 开源 | 进化算法 |
| Ray Tune | 开源 | 分布式黑盒搜索 |

求解器选择建议：
- 实验成本低：随机搜索、进化算法
- 实验成本高：贝叶斯优化
- 变量维度高：Nevergrad、CMA-ES、进化算法
- 需要并行实验：Ray Tune、Optuna

---

# 22. 贝叶斯优化 Bayesian Optimization

模型类型：
- Bayesian Optimization
- Active Learning for Experiment Design

特点：
- 适合实验次数少、单次实验成本高的场景
- 使用代理模型估计未知函数
- 用采集函数决定下一次实验
- 在探索和利用之间平衡
- 常用于实验参数优化和主动学习实验设计

现实例子：少量实验预算下寻找最佳 PCR 条件

实验室只有 30 次实验预算，需要优化退火温度、MgCl2 浓度、引物浓度和循环数，使扩增效率最高。

要决定：
- 下一组实验条件
- 是否探索不确定区域
- 是否利用当前预测最优区域

目标：
- 在有限实验次数内找到最佳条件
- 最大化实验响应
- 降低失败实验比例

变量：
- \(x_t\)：第 \(t\) 次实验条件
- \(y_t\)：第 \(t\) 次实验结果
- \(\mu(x)\)：代理模型预测均值
- \(\sigma(x)\)：代理模型预测不确定性
- \(\alpha(x)\)：采集函数

数学模型：

代理模型：

$$
y=f(x)+\epsilon
$$

下一点选择：

$$
x_{t+1}=\arg\max_{x\in\mathcal{X}}\alpha(x;\mathcal{D}_t)
$$

期望改进 EI：

$$
EI(x)=\mathbb{E}\left[\max(f(x)-f^+,0)\right]
$$

约束：

$$
l_i\le x_i\le u_i,\quad \forall i
$$

约束解释：
- 代理模型根据已有实验数据预测未知响应
- 采集函数综合考虑高预测值和高不确定性
- 参数必须在实验可行范围内

适用的实验室自动化场景：
- 实验参数优化
- 少量实验预算下寻找最佳实验条件
- 主动学习实验设计
- 试剂配方优化
- 高成本细胞实验优化

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| BoTorch | 开源 | 高级贝叶斯优化，基于 PyTorch |
| Ax | 开源 | 实验管理和 BO 平台 |
| scikit-optimize | 开源 | 简单易用 |
| Optuna | 开源 | 支持 TPE 等序贯优化 |

求解器选择建议：
- 入门和小规模：scikit-optimize、Optuna
- 高级约束、多目标、批量 BO：BoTorch、Ax
- 实验平台闭环：Ax + LIMS/自动化系统集成

---

# 23. 多目标优化 Multi-objective Optimization

模型类型：
- Multi-objective Optimization
- Pareto Optimization

特点：
- 同时优化多个互相冲突的目标
- 常见目标包括时间、成本、质量、风险、通量
- 输出通常不是单个解，而是 Pareto 前沿
- 可用加权和、ε-约束、进化算法、贝叶斯多目标优化
- 适合实验设计和自动化系统配置权衡

现实例子：同时优化时间、成本、质量和风险

实验室希望选择实验参数和调度策略，使总耗时短、成本低、检测质量高、失败风险低。

要决定：
- 实验参数
- 批次大小
- 设备分配
- QC 密度

目标：
- 时间最短
- 成本最低
- 质量最高
- 失败风险最低

变量：
- \(x\)：综合决策变量
- \(f_1(x)\)：时间
- \(f_2(x)\)：成本
- \(f_3(x)\)：质量损失
- \(f_4(x)\)：失败风险

数学模型：

加权和形式：

$$
\min \lambda_1 f_1(x)+\lambda_2 f_2(x)+\lambda_3 f_3(x)+\lambda_4 f_4(x)
$$

ε-约束形式：

$$
\min f_1(x)
$$

$$
f_2(x)\le \epsilon_2,\quad f_3(x)\le \epsilon_3,\quad f_4(x)\le \epsilon_4
$$

约束：

$$
x\in \mathcal{X}
$$

约束解释：
- \(\mathcal{X}\) 表示所有设备、容量、合规和实验安全约束
- 加权和需要人为设置权重
- ε-约束可以生成多个 Pareto 解

适用的实验室自动化场景：
- 同时优化时间、成本、质量、风险
- 实验性能和成本权衡
- 多目标实验设计
- 设备配置投资决策
- QC 密度与通量权衡

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| pymoo | 开源 | 多目标进化算法丰富 |
| DEAP | 开源 | 自定义进化算法 |
| BoTorch | 开源 | 多目标贝叶斯优化 |
| Gurobi | 非开源/商业 | 多目标 MILP 支持 |

求解器选择建议：
- 线性/MILP 多目标：Gurobi、CPLEX、加权和
- 黑盒多目标：pymoo、DEAP、BoTorch
- 实验预算少：多目标贝叶斯优化
- 决策解释性强：生成 Pareto 前沿供专家选择

---

# 24. DAG / Workflow Scheduling

模型类型：
- DAG Scheduling
- Workflow Scheduling
- Resource Scheduling

特点：
- 任务之间形成有向无环图 DAG
- 任务必须等待上游任务完成
- 资源包括 CPU、GPU、内存、存储、许可证
- 支持失败重试、缓存、动态扩缩容
- 适合数据分析 pipeline 和实验流程 DAG 调度

现实例子：测序数据分析 pipeline 调度

测序数据需要经过质控、比对、去重、变异检测、注释和报告生成。不同步骤依赖上游结果，并消耗不同 CPU/GPU/内存资源。

要决定：
- 每个任务在哪个计算节点执行
- 每个任务何时开始
- 失败任务如何重试
- 多个样本 pipeline 如何并行

目标：
- Pipeline 总完成时间最小
- 计算资源利用率最大
- 失败恢复时间最短
- 成本最低

变量：
- \(s_i\)：任务 \(i\) 的开始时间
- \(C_i\)：任务 \(i\) 的完成时间
- \(x_{ik}\)：任务 \(i\) 是否分配到资源节点 \(k\)
- \(r_{im}\)：任务 \(i\) 对资源 \(m\) 的需求

数学模型：

$$
\min C_{\max}
$$

约束：

$$
C_i=s_i+p_i,\quad \forall i
$$

DAG 依赖：

$$
s_j\ge C_i,\quad \forall (i,j)\in E
$$

资源容量：

$$
\sum_{i:t\in[s_i,C_i)} r_{im}x_{ik}\le R_{km},\quad \forall k,m,t
$$

节点分配：

$$
\sum_k x_{ik}=1,\quad \forall i
$$

约束解释：
- 下游任务必须等待上游任务完成
- 每个任务必须分配到一个可用计算节点
- CPU/GPU/内存等资源不能超限
- 失败重试可建模为额外任务或重试策略

适用的实验室自动化场景：
- 数据分析 pipeline 调度
- 实验流程 DAG 调度
- CPU/GPU/内存资源约束下的分析任务调度
- 多样本批量分析
- 失败重试和自动恢复

常见求解器：

| 求解器/工具 | 是否开源 | 适用原因 |
|---|---|---|
| Airflow | 开源 | 通用 DAG 编排 |
| Prefect | 开源/商业版本并存 | 现代工作流编排 |
| Dagster | 开源/商业版本并存 | 数据资产和 pipeline 管理 |
| Nextflow | 开源 | 生信工作流常用 |
| Snakemake | 开源 | 科研和生信 pipeline 常用 |

求解器选择建议：
- 生产数据 pipeline：Airflow、Dagster、Prefect
- 生信分析：Nextflow、Snakemake
- 需要严格最优资源调度：可结合 CP-SAT/MILP
- 需要失败重试：选择具备 retry、checkpoint、cache 的工作流系统

---

# 25. 求解器与建模工具总览

| 求解器/工具 | 是否开源 | 主要适用问题 | 支持模型 | Python 接口 | 备注 |
|---|---|---|---|---|---|
| HiGHS | 开源 | LP、MILP | LP/MIP | 有 | 高性能开源求解器 |
| SciPy linprog | 开源 | 小中规模 LP | LP | 有 | 底层可调用 HiGHS |
| CBC | 开源 | ILP、MILP | LP/MIP | 有 | PuLP 常用后端 |
| GLPK | 开源 | LP、ILP | LP/MIP | 有 | 教学和小规模应用 |
| SCIP | 学术免费/部分商业限制 | MILP、MINLP、组合优化 | MIP、MINLP | 有 | 商业使用需核查许可 |
| Gurobi | 非开源/商业 | 大规模 LP/MILP/QP | LP、MIP、QP、MIQP | 有 | 商业性能强 |
| CPLEX | 非开源/商业 | 大规模 LP/MILP/调度 | LP、MIP、QP | 有 | IBM 商业求解器 |
| MOSEK | 非开源/商业 | 凸优化、锥优化、QP | LP、QP、SOCP、SDP | 有 | 凸优化强 |
| Xpress | 非开源/商业 | 大规模数学规划 | LP、MIP、QP | 有 | FICO 商业求解器 |
| PuLP | 开源 | LP/MILP 建模 | LP/MIP | 有 | 易学，适合原型 |
| Pyomo | 开源 | 通用优化建模 | LP/MIP/NLP | 有 | 学术和工业常用 |
| CVXPY | 开源 | 凸优化建模 | LP/QP/SOCP/SDP | 有 | 凸优化表达自然 |
| OR-Tools Linear Solver | 开源 | LP/MIP | LP/MIP | 有 | Google OR-Tools 组件 |
| JuMP | 开源，Julia 生态 | 通用数学建模 | LP/MIP/NLP | 非 Python，Julia | Julia 生态高性能建模 |
| Google OR-Tools CP-SAT | 开源 | CSP、调度、装箱、指派 | CP-SAT | 有 | 实验室排程常用 |
| Timefold | 开源/商业版本并存 | 业务排程、资源调度 | 约束规划/启发式 | 有/主要 Java 生态 | OptaPlanner 后续生态之一 |
| OptaPlanner | 开源，需说明生态变化 | 业务排程 | 约束规划/启发式 | 主要 Java | 原 Red Hat 项目，生态已变化 |
| IBM CP Optimizer | 非开源/商业 | 高级调度、CP | CP、Scheduling | 有 | 工业调度强 |
| OR-Tools Routing | 开源 | TSP、VRP、Routing | 路径优化 | 有 | VRP 时间窗、容量支持好 |
| NetworkX | 开源 | 图算法、网络流、原型 | Graph Algorithms | 有 | 适合原型和分析 |
| LKH | 通常可免费用于研究，许可需核查 | TSP、VRP 启发式 | Routing | 可间接调用 | 高质量路径启发式 |
| BoTorch | 开源 | 贝叶斯优化 | BO | 有 | 高级 BO，支持多目标 |
| Ax | 开源 | 实验设计、BO 平台 | BO | 有 | 与 BoTorch 集成 |
| scikit-optimize | 开源 | 简单贝叶斯优化 | BO | 有 | 易用但维护活跃度需关注 |
| Nevergrad | 开源 | 黑盒优化 | Derivative-free | 有 | 算法丰富 |
| Optuna | 开源 | 黑盒优化、参数搜索 | TPE、随机、进化 | 有 | 易集成 |
| Ray Tune | 开源 | 分布式搜索 | HPO/Black-box | 有 | 适合并行实验 |
| pymoo | 开源 | 多目标优化 | MOEA | 有 | NSGA-II 等 |
| DEAP | 开源 | 进化算法 | GA/GP/EA | 有 | 灵活 |
| Airflow | 开源 | DAG 工作流 | Workflow | 有 | 生产编排常用 |
| Prefect | 开源/商业版本并存 | Workflow | DAG/Flow | 有 | 易用，云服务可选 |
| Dagster | 开源/商业版本并存 | 数据 pipeline | DAG/Assets | 有 | 数据资产管理强 |
| Luigi | 开源 | 批处理 pipeline | DAG | 有 | Spotify 开源 |
| Nextflow | 开源 | 生信 workflow | DAG | 可通过命令行 | 生信常用 |
| Snakemake | 开源 | 科研 workflow | DAG | Python 风格 | 生信和科研常用 |

---

# 26. 求解器选型建议

| 场景 | 推荐求解器/工具 | 原因 |
|---|---|---|
| 小规模 LP | SciPy linprog、HiGHS | 易用、开源、速度快 |
| 大规模 LP | HiGHS、Gurobi、CPLEX | HiGHS 开源性能好，商业求解器更稳健 |
| 小规模 ILP / MILP | PuLP + CBC、Pyomo + HiGHS | 建模简单，开源可用 |
| 大规模 MILP | Gurobi、CPLEX、SCIP | 商业求解器性能强，SCIP 组合优化强 |
| 指派问题 | SciPy linear_sum_assignment、OR-Tools | 标准指派可快速求解，复杂约束用 OR-Tools |
| 装箱与批次划分 | OR-Tools CP-SAT、SCIP、Gurobi | 组合约束多，CP-SAT 表达自然 |
| 孔板布局 CSP | OR-Tools CP-SAT、MiniZinc | 适合相邻、隔离、固定间隔等规则 |
| 复杂调度问题 | OR-Tools CP-SAT、IBM CP Optimizer、Timefold | 调度约束表达能力强 |
| Job Shop / Flow Shop | OR-Tools CP-SAT、IBM CP Optimizer | Interval 和 no-overlap 建模方便 |
| 样本-设备分配 | MILP：HiGHS/CBC/Gurobi；或 OR-Tools | 简单分配用 MILP，规则复杂用 CP-SAT |
| TSP / VRP / 移液路径 | OR-Tools Routing、LKH、NetworkX | Routing 专用能力强 |
| 机械臂转运路由 | OR-Tools Routing + CP-SAT | 路径和时间窗结合，必要时与调度联合 |
| 库存优化 | Pyomo + HiGHS/Gurobi、SciPy | 多周期库存适合数学规划 |
| 随机需求下库存优化 | Pyomo、Gurobi、CPLEX、仿真优化 | 场景规划或仿真评估 |
| 自动化产线通量优化 | NetworkX、OR-Tools、SimPy + Optuna | 网络流看理论通量，仿真看真实排队 |
| 贝叶斯优化实验设计 | BoTorch、Ax、scikit-optimize、Optuna | 实验预算少时样本效率高 |
| 黑盒配方优化 | Optuna、Nevergrad、DEAP、Ray Tune | 无显式公式，适合无导数搜索 |
| 多目标实验设计 | pymoo、BoTorch、DEAP、Gurobi | Pareto 前沿和权衡分析 |
| 故障情况下实时重排程 | OR-Tools CP-SAT、Timefold、启发式 | 支持限时求解和局部调整 |
| 合规质控规则检查 | OR-Tools CP-SAT、规则引擎、MiniZinc | 硬约束和软约束表达方便 |
| 数据分析 pipeline 调度 | Airflow、Prefect、Dagster、Nextflow、Snakemake | 支持 DAG、重试、依赖和资源管理 |
| 开源优先方案 | HiGHS、CBC、OR-Tools、Pyomo、Optuna、Airflow | 生态成熟，无商业授权成本 |
| 商业高性能方案 | Gurobi、CPLEX、MOSEK、Xpress、IBM CP Optimizer | 大规模、稳定性和技术支持更强 |
| 原型验证阶段 | PuLP、Pyomo、SciPy、NetworkX、OR-Tools | 上手快，便于模型迭代 |
| 生产部署阶段 | Gurobi/CPLEX/OR-Tools/Timefold/Airflow | 稳定性、性能、可集成性更重要 |

---

# 27. 实验室自动化优化建模的一般流程

## 27.1 明确实验流程

首先要把实验室自动化系统拆解清楚：

- 样本：样本数量、优先级、截止时间、兼容性、复测规则
- 任务：加样、孵育、PCR、离心、读板、LC-MS、数据分析
- 设备：PCR 仪、离心机、读板仪、机械臂、液体处理工作站
- 耗材：枪头、孔板、试管、试剂盒
- 试剂：库存、有效期、冷链容量、批号
- 计算资源：CPU、GPU、内存、存储、许可证

## 27.2 明确优化目标

常见目标包括：

- 时间：总完成时间最小、延迟最小、等待时间最小
- 成本：试剂成本、设备启用成本、人力成本、计算成本
- 通量：单位时间完成样本数最大
- 质量：批次效应最小、QC 合格率最高、误差最低
- 风险：失败风险、污染风险、缺货风险、设备故障风险
- 合规性：满足 SOP、GxP、审计追踪和质控规则

如果目标冲突，应考虑多目标优化或分层优化。

## 27.3 定义决策变量

常见变量类型：

- 排序变量：任务 \(i\) 是否在任务 \(j\) 前执行
- 分配变量：样本 \(i\) 是否分配给设备 \(j\)
- 路径变量：机械臂是否从节点 \(i\) 移动到节点 \(j\)
- 时间变量：任务开始时间 \(s_i\)、完成时间 \(C_i\)
- 库存变量：订货量 \(q_t\)、库存量 \(I_t\)
- 参数变量：温度、浓度、pH、反应时间
- 逻辑变量：是否启用设备、是否开启批次、是否复测

## 27.4 建立约束条件

常见约束包括：

- 设备容量：设备同一时间只能处理有限任务
- 时间窗口：样本必须在截止时间前完成
- 前后依赖：PCR 前必须完成提取，读板前必须完成孵育
- 兼容性：某些样本不能使用某些设备或孔位
- 库存：试剂和耗材必须足够
- QC：阳性、阴性、空白、重复样本必须按规则布置
- 法规：记录、批号、审计追踪和 SOP 约束
- 路径：机械臂或移液枪移动路径不能冲突
- 资源：CPU/GPU/内存/许可证不能超限

## 27.5 选择模型类型

可按问题特征选择：

| 问题特征 | 推荐模型 |
|---|---|
| 连续采购量或资源分配 | LP |
| 是否选择、是否启用 | ILP / 0-1 规划 |
| 分配 + 容量 + 固定成本 | MILP |
| 样本到设备或孔位的匹配 | Assignment |
| 批次划分、孔板装载 | Bin Packing |
| 孔板布局和规则检查 | CSP / CP-SAT |
| 今日任务排序 | Scheduling |
| 多步骤工艺流程 | Job Shop / RCPSP |
| 移液路径和机械臂访问顺序 | TSP / VRP / Routing |
| 产线通量和瓶颈 | Network Flow / Simulation |
| 试剂库存和补货 | Inventory Optimization |
| 需求、故障、时间波动 | Stochastic / Robust Optimization |
| 动态插单和故障恢复 | Online / Rolling Horizon |
| 配方和参数连续优化 | NLP |
| 无显式公式实验优化 | Black-box / Bayesian Optimization |
| 时间、成本、质量权衡 | Multi-objective Optimization |
| 数据分析流程 | DAG / Workflow Scheduling |

## 27.6 选择求解器

选择求解器时考虑：

- 问题规模：变量数、约束数、时间窗口长度
- 是否实时：秒级重排程还是离线计划
- 是否需要全局最优：生产系统有时可接受近似解
- 是否开源优先：预算、部署和许可要求
- 是否需要商业支持：稳定性、性能、服务等级
- 是否包含复杂逻辑：CP-SAT 往往比 MILP 更自然
- 是否是黑盒：选择 Optuna、Nevergrad、BoTorch 等

## 27.7 原型验证

建议先用小规模数据验证：

- 变量定义是否正确
- 约束是否过严导致无解
- 目标是否符合业务直觉
- 解是否违反 SOP 或实验常识
- 边界情况是否合理，例如设备全故障、试剂不足、急诊插队

## 27.8 接入自动化系统

优化模型需要与实际系统集成：

- LIMS：样本信息、优先级、检测项目
- ELN：实验方案和记录
- 调度系统：任务下发和状态回传
- 机器人控制系统：机械臂、移液工作站、自动仓储
- 仪器系统：PCR、LC-MS、读板仪、测序仪
- 数据 pipeline：分析任务、结果回传、报告生成
- 库存系统：试剂、耗材、批号、有效期

## 27.9 滚动优化与异常恢复

生产环境中必须处理动态事件：

- 动态插单：急诊样本、高优先级样本
- 设备故障：冻结已执行任务，重排未开始任务
- 试剂缺货：替代试剂、调整批次、延迟低优先级任务
- 样本失败：自动生成复测任务
- 数据分析失败：重试、换节点、恢复缓存
- 维护窗口：提前预留设备停机时间

推荐架构：

- 战略层：每日或每周全局计划
- 战术层：小时级滚动优化
- 执行层：分钟级规则修复和异常处理

## 27.10 持续监控和模型迭代

上线后持续监控：

- 吞吐量：每天完成样本数
- 延迟：按时完成率和逾期样本数
- 设备利用率：瓶颈设备识别
- 队列长度：各工位等待样本数
- 实验失败率：复测率、QC 失败率
- 成本：试剂、耗材、设备、人力、计算成本
- 模型质量：预测处理时间与实际处理时间偏差
- 稳定性：重排程频率和计划扰动程度

优化模型应持续迭代：

- 用历史数据更新处理时间和失败率
- 用仿真验证新设备配置
- 用贝叶斯优化改进实验参数
- 用多目标优化支持管理层权衡决策
- 用在线优化增强异常恢复能力


In [1]:
from ortools.linear_solver import pywraplp

In [17]:
# Create the mip solver with the CP-SAT backend.
solver = pywraplp.Solver.CreateSolver("GUROBI")
if not solver:
    print("solver is unavailable")

solver is unavailable


In [16]:
infinity = solver.infinity()
# x and y are integer non-negative variables.
x = solver.IntVar(0.0, infinity, "x")
y = solver.IntVar(0.0, infinity, "y")

print("Number of variables =", solver.NumVariables())

AttributeError: 'NoneType' object has no attribute 'infinity'

In [ ]:
# x + 7 * y <= 17.5.
solver.Add(x + 7 * y <= 17.5)

# x <= 3.5.
solver.Add(x <= 3.5)

print("Number of constraints =", solver.NumConstraints())

Number of constraints = 2


In [13]:
# Maximize x + 10 * y.
solver.Maximize(x + 10 * y)

In [14]:
print(f"Solving with {solver.SolverVersion()}")
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    print("Solution:")
    print("Objective value =", solver.Objective().Value())
    print("x =", x.solution_value())
    print("y =", y.solution_value())
else:
    print("The problem does not have an optimal solution.")

Solving with CP-SAT solver v9.15.6755
Solution:
Objective value = 23.0
x = 3.0
y = 2.0
